### Udaplay_01_solution_project — load, process, and format the provided game JSON files.

This solution loads, processes, and formats the games.json file

## 1 Setting up dependencies

import the necessary packages and libraries and load environment

In [1]:
from dotenv import load_dotenv
import os

load_dotenv("config.env")
assert os.getenv("OPENAI_API_KEY") is not None
assert os.getenv("TAVILY_API_KEY") is not None

# 2 RAG and Long Term Memory initialization


In [3]:
import json
from lib.llm import LLM
import os
from lib.vector_db import VectorStoreManager
from lib.documents import Document, Corpus
from lib.memory import LongTermMemory

rag_llm = LLM(model="gpt-4o-mini", temperature=0.3)

CHROMA_PERSIST_DIR = os.path.join(os.path.abspath("."), "chroma_db")
vector_store_manager = VectorStoreManager(
    os.getenv("OPENAI_API_KEY"), persist_directory=CHROMA_PERSIST_DIR
)
long_term_memory = LongTermMemory(vector_store_manager)


def _load_game_documents():
    games_dir = os.path.join(os.path.abspath("."), "games")
    documents = []
    for filename in sorted(os.listdir(games_dir)):
        if filename.endswith(".json"):
            with open(os.path.join(games_dir, filename), "r", encoding="utf-8") as f:
                g = json.load(f)
            documents.append(
                Document(
                    content=(
                        f"{g['Name']} is available on {g['Platform']}. "
                        f"Genre: {g.get('Genre', 'Unknown')}. "
                        f"Publisher: {g.get('Publisher', 'Unknown')}. "
                        f"Year of release: {g.get('YearOfRelease', 'Unknown')}. "
                        f"{g.get('Description', '')}"
                    ),
                    metadata={
                        "game": g["Name"],
                        "platform": g["Platform"],
                        "genre": g.get("Genre", ""),
                        "publisher": g.get("Publisher", ""),
                        "year": g.get("YearOfRelease", ""),
                    },
                )
            )
    return Corpus(documents)



long_term_memory.vector_store.add(_load_game_documents())
print(f"Loaded {len(_load_game_documents())} game documents into the vector store.")

Loaded 15 game documents into the vector store.


# 3 Query the vector database

In [6]:

# --- Collection size ---
total = long_term_memory.vector_store._collection.count()
print(f"Total documents in collection: {total}\n")

# --- Semantic search demo ---
QUERY = "open world adventure with exploration"
results = long_term_memory.vector_store.query(query_texts=[QUERY], n_results=5)

print(f'Query: "{QUERY}"\n')
print(f"{'#':<4} {'Game':<30} {'Platform':<20} {'Genre':<15} {'Year':>4} {'Dist':>8}")
print("-" * 90)

documents = results["documents"][0]
metadatas = results["metadatas"][0]
distances = results["distances"][0]

for rank, (doc, meta, dist) in enumerate(zip(documents, metadatas, distances), start=1):
    game = meta.get("game", "N/A")
    platform = meta.get("platform", "N/A")
    genre = meta.get("genre", "N/A")
    year = meta.get("year", "N/A")
    print(f"{rank:<4} {game:<30} {platform:<20} {genre:<15} {str(year):>4} {dist:>8.4f}")


Total documents in collection: 15

Query: "open world adventure with exploration"

#    Game                           Platform             Genre           Year     Dist
------------------------------------------------------------------------------------------
1    Marvel's Spider-Man            PlayStation 4        Action-adventure 2018   0.1929
2    Minecraft                      Xbox One             Sandbox, Survival 2014   0.1950
3    Grand Theft Auto: San Andreas  PlayStation 2        Action-adventure 2004   0.1991
4    Kinect Adventures!             Xbox 360             Party           2010   0.2062
5    Super Mario World              Super Nintendo Entertainment System (SNES) Platformer      1990   0.2065
